# V2 Demographics Features

## 목적

서울 25개 구의 **인구·세대 변화, 인구 이동, 아파트 매입자 구성**을 구×월 Feature로 만들고,
06 Merge에서 쓸 **구별 주택·아파트 재고(분모)**를 준비한다. 모든 값은 그 시점에 공개돼 있던 정보만 쓴다.

### 주요 데이터 (KOSIS, `05a_collect_kosis_demographics.py`)

| 표 | 내용 | 공표 | 사용 시점 |
|---|---|---|---|
| 101/DT_1B040A3, DT_1B040B3 | 주민등록 인구·세대수 (월) | M월분 → M+1월 초 | **M+2월 행부터** |
| 101/DT_1B04006 | 1세별 주민등록인구 (월) | 같음 | M+2월 행부터 |
| 101/DT_1B26001_A01 | 시군구별 이동자수 (월) | M월분 → M+1월 하순 | M+2월 행부터 |
| 408/DT_408_2006_S0065, S0077 | 매입자 거주지별·연령대별 아파트 매매 (한국부동산원, 월) | M월분 → M+1월 | M+2월 행부터 |
| 101/DT_1JU1501 | 주택총조사 주택·아파트 수 (연, 11월 1일 기준) | t년분 → t+1년 7월 하순 | **t+1년 8월 행부터** |

### 기본 원칙

- **수준값은 모델에 넣지 않는다.** 인구·세대·재고·연령 비중의 현재 수준은 5년 동안 거의 변하지 않아
  모델이 구를 구분하는 ID처럼 쓰게 된다(V1 회귀의 위치 외우기 문제). 모델에는 증감·이동률·매입자 비중만 쓰고,
  수준값은 웹 표시와 06 Merge의 분모로만 저장한다.
- 한국부동산원 매입자 통계는 KOSIS에서 전체 시계열이 2026-09-11에 다시 갱신돼 최초 공표값을 구할 수 없다.
  건수 대신 **비중**만 써서 사후 수정 영향을 줄인다.

In [1]:
# ============================================================
# 경로·상수
# ============================================================

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", message="The 'generic' unit for NumPy timedelta")


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists():
            return path
    raise FileNotFoundError("프로젝트 루트에서 data 폴더를 찾을 수 없습니다.")


PROJECT_DIR = find_project_root()
KOSIS_DIR = PROJECT_DIR / "data" / "external" / "kosis"
PROCESSED_DIR = PROJECT_DIR / "data" / "v2_5year_seoul" / "processed"

PUBLISH_LAG = 2           # 데이터월 M → M+2월 행부터
CENSUS_RELEASE_MONTH = 8  # 주택총조사 t년분 → t+1년 8월 행부터

region_month = pd.read_parquet(PROCESSED_DIR / "market_leading_v2.parquet", columns=["기준월", "자치구", "권역"])
ALL_GU = sorted(region_month["자치구"].unique())
MODEL_MONTHS = pd.PeriodIndex(region_month["기준월"].unique(), freq="M")

raw = {p.stem: pd.read_parquet(p) for p in sorted(KOSIS_DIR.glob("*.parquet"))}
for name, d in raw.items():
    print(f"{name:20s} {d.shape}  {d['PRD_DE'].min()} ~ {d['PRD_DE'].max()}")

DT_1B04006           (278460, 22)  201801 ~ 202609
DT_1B040A3           (2730, 18)  201801 ~ 202609
DT_1B040B3           (2730, 18)  201801 ~ 202609
DT_1B26001_A01       (21632, 18)  201801 ~ 202608
DT_1JU1501           (572, 18)  2015 ~ 2025
DT_408_2006_S0065    (27040, 18)  201801 ~ 202608
DT_408_2006_S0077    (38272, 18)  201901 ~ 202608


## 1. 원본 정리 (월 × 지역)

In [2]:
# ============================================================
# 표별 정리: 지역명은 '서울' + 25개 구로 통일
# ============================================================

def region_name(s):
    return s.replace({"서울특별시": "서울"})


def monthly(d, value_name, **filters):
    x = d.copy()
    for col, val in filters.items():
        x = x[x[col].eq(val)]
    x = x.assign(
        지역=region_name(x["C1_NM"]),
        data_month=pd.PeriodIndex(x["PRD_DE"], freq="M"),
        value=pd.to_numeric(x["DT"], errors="coerce"),
    )
    s = x.groupby(["지역", "data_month"])["value"].sum().rename(value_name)
    return s


# 주민등록 인구·세대
pop = monthly(raw["DT_1B040A3"], "인구", ITM_ID="T20")
hh = monthly(raw["DT_1B040B3"], "세대", ITM_ID="T1")

# 연령별 인구: '30세', '100세 이상' 같은 이름에서 나이를 읽는다
age = raw["DT_1B04006"].copy()
age = age[age["C2_NM"].ne("계")]
age["나이"] = age["C2_NM"].str.extract(r"(\d+)")[0].astype(int)
age_total = monthly(age, "연령합계")
pop_3044 = monthly(age[age["나이"].between(30, 44)], "인구_30_44")
pop_65 = monthly(age[age["나이"] >= 65], "인구_65이상")

# 인구 이동
mig = raw["DT_1B26001_A01"]
mig_items = {"T10": "총전입", "T20": "총전출", "T25": "순이동", "T31": "타구전입", "T32": "타구전출"}
migration = pd.concat([monthly(mig, name, ITM_ID=code) for code, name in mig_items.items()], axis=1)

# 매입자 거주지·연령대 (호수)
buyer_res = raw["DT_408_2006_S0065"]
buyer_res = buyer_res[buyer_res["ITM_NM"].eq("호수")]
buyer = pd.concat([
    monthly(buyer_res, "매입_계", C2_NM="매입자거주지"),
    monthly(buyer_res, "매입_구내", C2_NM="관할 시군구내"),
    monthly(buyer_res, "매입_서울타구", C2_NM="관할 시도내"),
    monthly(buyer_res, "매입_서울외", C2_NM="관할 시도외 기타"),
], axis=1)

buyer_age = raw["DT_408_2006_S0077"]
buyer_age = buyer_age[buyer_age["ITM_NM"].eq("호수")]
buyer = buyer.join(pd.concat([
    monthly(buyer_age, "매입_연령계", C2_NM="매입자연령대"),
    monthly(buyer_age, "매입_30대", C2_NM="30대"),
    monthly(buyer_age, "매입_40대", C2_NM="40대"),
], axis=1), how="left")

base = pd.concat([pop, hh, age_total, pop_3044, pop_65, migration, buyer], axis=1).sort_index()
print(base.shape)
print("지역:", sorted(base.index.get_level_values("지역").unique()) == sorted(ALL_GU + ["서울"]))
display(base.xs("강남구").loc["2021-06":"2021-09"])

(2730, 17)
지역: True


,인구,세대,연령합계,인구_30_44,인구_65이상,총전입,총전출,순이동,타구전입,타구전출,매입_계,매입_구내,매입_서울타구,매입_서울외,매입_연령계,매입_30대,매입_40대
data_month,,,,,,,,,,,,,,,,,
2021-06,531375,232707,531375,126225,76028,6222.0,6828.0,-606.0,2342.0,2764.0,201.0,97.0,59.0,45.0,201.0,44.0,93.0
2021-07,530529,232642,530529,125737,76348,6718.0,7633.0,-915.0,2648.0,3299.0,313.0,205.0,64.0,44.0,313.0,70.0,127.0
2021-08,532818,233815,532818,125868,76672,11556.0,9317.0,2239.0,5165.0,3355.0,380.0,240.0,93.0,47.0,380.0,90.0,126.0
2021-09,533359,234218,533359,125786,77072,8072.0,7548.0,524.0,3403.0,2823.0,269.0,155.0,64.0,50.0,269.0,50.0,106.0


In [3]:
# ============================================================
# 정합성 점검
# - 1세별 인구 합계 = 주민등록 총인구
# - 25개 구 합 = 서울 (인구, 세대, 매입 건수)
# - 매입자 거주지 합 = 매입 계, 연령대 합계 = 거주지 합계
# ============================================================

gap_age = (base["연령합계"] - base["인구"]).abs().max()
print("1세별 합계 vs 총인구 최대 차이:", gap_age)
assert gap_age == 0

gu_sum = base.drop(index="서울", level="지역").groupby(level="data_month").sum()
seoul = base.xs("서울")
for col in ["인구", "세대", "매입_계"]:
    gap = (gu_sum[col] - seoul[col]).abs().max()
    print(f"{col}: 25개 구 합 vs 서울 최대 차이 {gap}")
    assert gap == 0

parts = base[["매입_구내", "매입_서울타구", "매입_서울외"]].sum(axis=1)
print("매입자 거주지 합 vs 계 최대 차이:", (parts - base["매입_계"]).abs().max())
print("연령대 계 vs 거주지 계 최대 차이:", (base["매입_연령계"] - base["매입_계"]).abs().max())

1세별 합계 vs 총인구 최대 차이: 0
인구: 25개 구 합 vs 서울 최대 차이 0
세대: 25개 구 합 vs 서울 최대 차이 0
매입_계: 25개 구 합 vs 서울 최대 차이 0.0
매입자 거주지 합 vs 계 최대 차이: 0.0
연령대 계 vs 거주지 계 최대 차이: 0.0


In [4]:
# ============================================================
# 사후 수정 흔적: KOSIS 최종 수정일(LST_CHN_DE) — 데이터 연도별 범위
# ============================================================

rows = []
for name in ["DT_1B040A3", "DT_1B26001_A01", "DT_408_2006_S0065", "DT_1JU1501"]:
    d = raw[name].assign(데이터연도=raw[name]["PRD_DE"].str[:4])
    g = d.groupby("데이터연도")["LST_CHN_DE"].agg(["min", "max"]).loc["2020":]
    g.insert(0, "표", name)
    rows.append(g.reset_index())
display(pd.concat(rows, ignore_index=True))

,데이터연도,표,min,max
0,2020,DT_1B040A3,2025-03-12,2025-03-12
1,2021,DT_1B040A3,2025-03-12,2025-03-12
2,2022,DT_1B040A3,2025-03-12,2025-03-12
3,2023,DT_1B040A3,2025-03-12,2025-03-12
4,2024,DT_1B040A3,2025-03-12,2025-03-12
5,2025,DT_1B040A3,2025-03-12,2026-01-05
6,2026,DT_1B040A3,2026-02-03,2026-10-02
7,2020,DT_1B26001_A01,2020-02-11,2021-01-06
8,2021,DT_1B26001_A01,2021-03-08,2022-01-06
9,2022,DT_1B26001_A01,2022-02-21,2023-01-19


## 2. 구×월 Feature (`demographics_v2.parquet`)

데이터월 d 기준으로 계산한 뒤 **d+2월 행**에 붙인다.

| Feature | 정의 | 용도 |
|---|---|---|
| `인구_3m증감률`, `인구_12m증감률`, `세대_12m증감률` | 주민등록 인구·세대 변화율 | 모델 |
| `세대원수_12m변화` | (인구 ÷ 세대)의 12개월 변화 | 모델 |
| `비중_30_44세_12m변화`, `비중_65세이상_12m변화` | 연령 비중의 12개월 변화(%p) | 모델 |
| `순이동률_3m/6m/12m` | 순이동 합 ÷ 인구 (%) | 모델 |
| `타구전입_비중_3m` | 서울 다른 구에서 온 전입 ÷ 총전입 (3개월) | 모델 |
| `외지인매입_비중_3m`, `서울외매입_비중_3m` | 구 밖 / 서울 밖 거주자 매입 ÷ 전체 매입 (3개월) | 모델 |
| `30대매입_비중_3m` | 30대 매입 ÷ 전체 매입 (3개월) | 모델 |
| `서울_순이동률_12m`, `서울_외지인매입_비중_3m` | 서울 전체 값 (`서울외매입` 기준) | 모델 |
| `인구`, `세대`, `세대원수`, `비중_30_44세`, `비중_65세이상` | 수준값 | 표시·분모 (모델 제외) |

In [5]:
# ============================================================
# 데이터월 기준 Feature
# ============================================================

def compute_features(base):
    f = pd.DataFrame(index=base.index)
    g = lambda s, fn: s.groupby(level="지역", group_keys=False).apply(fn)
    shift = lambda col, k: g(base[col], lambda x: x.shift(k))
    rsum = lambda col, w: g(base[col], lambda x: x.rolling(w, min_periods=w).sum())

    f["인구_3m증감률"] = (base["인구"] / shift("인구", 3) - 1) * 100
    f["인구_12m증감률"] = (base["인구"] / shift("인구", 12) - 1) * 100
    f["세대_12m증감률"] = (base["세대"] / shift("세대", 12) - 1) * 100

    size = base["인구"] / base["세대"]
    f["세대원수_12m변화"] = size - g(size, lambda x: x.shift(12))

    share_3044 = base["인구_30_44"] / base["인구"] * 100
    share_65 = base["인구_65이상"] / base["인구"] * 100
    f["비중_30_44세_12m변화"] = share_3044 - g(share_3044, lambda x: x.shift(12))
    f["비중_65세이상_12m변화"] = share_65 - g(share_65, lambda x: x.shift(12))

    for w in (3, 6, 12):
        f[f"순이동률_{w}m"] = rsum("순이동", w) / base["인구"] * 100

    f["타구전입_비중_3m"] = rsum("타구전입", 3) / rsum("총전입", 3)
    f["외지인매입_비중_3m"] = (rsum("매입_서울타구", 3) + rsum("매입_서울외", 3)) / rsum("매입_계", 3)
    f["서울외매입_비중_3m"] = rsum("매입_서울외", 3) / rsum("매입_계", 3)
    f["30대매입_비중_3m"] = rsum("매입_30대", 3) / rsum("매입_연령계", 3)

    # 표시·분모용 수준값
    f["인구"] = base["인구"]
    f["세대"] = base["세대"]
    f["세대원수"] = size
    f["비중_30_44세"] = share_3044
    f["비중_65세이상"] = share_65
    return f


MODEL_FEATURES = [
    "인구_3m증감률", "인구_12m증감률", "세대_12m증감률", "세대원수_12m변화",
    "비중_30_44세_12m변화", "비중_65세이상_12m변화",
    "순이동률_3m", "순이동률_6m", "순이동률_12m", "타구전입_비중_3m",
    "외지인매입_비중_3m", "서울외매입_비중_3m", "30대매입_비중_3m",
    "서울_순이동률_12m", "서울_외지인매입_비중_3m",
]
LEVEL_FEATURES = ["인구", "세대", "세대원수", "비중_30_44세", "비중_65세이상"]


def build_demographics(base, months, lag=PUBLISH_LAG):
    f = compute_features(base).reset_index()
    f["month"] = f["data_month"] + lag

    seoul = f[f["지역"].eq("서울")].set_index("month")
    gu = f[f["지역"].ne("서울")].rename(columns={"지역": "자치구"})
    gu = gu[gu["month"].isin(months)].copy()
    gu["서울_순이동률_12m"] = gu["month"].map(seoul["순이동률_12m"])
    gu["서울_외지인매입_비중_3m"] = gu["month"].map(seoul["서울외매입_비중_3m"])
    return gu[["자치구", "month", "data_month"] + MODEL_FEATURES + LEVEL_FEATURES]


demo_rm = build_demographics(base, MODEL_MONTHS)
print(demo_rm.shape)
display(demo_rm[MODEL_FEATURES].describe().T.round(3))

(1500, 23)


,count,mean,std,min,25%,50%,75%,max
인구_3m증감률,1500.0,-0.193,0.427,-2.259,-0.373,-0.227,-0.069,5.066
인구_12m증감률,1500.0,-0.868,1.272,-5.270,-1.466,-1.001,-0.435,7.659
세대_12m증감률,1500.0,0.363,1.226,-5.122,-0.241,0.216,0.865,7.598
세대원수_12m변화,1500.0,-0.026,0.014,-0.071,-0.035,-0.023,-0.018,0.007
비중_30_44세_12m변화,1500.0,0.073,0.246,-0.491,-0.106,0.043,0.256,0.914
비중_65세이상_12m변화,1500.0,0.857,0.290,-0.126,0.664,0.864,1.033,1.689
순이동률_3m,1500.0,-0.134,0.386,-2.279,-0.309,-0.193,-0.041,4.807
순이동률_6m,1500.0,-0.282,0.672,-3.773,-0.596,-0.379,-0.103,6.135
순이동률_12m,1500.0,-0.603,1.160,-5.374,-1.151,-0.777,-0.254,7.051
타구전입_비중_3m,1500.0,0.409,0.053,0.286,0.367,0.414,0.441,0.550


## 3. 구별 주택·아파트 재고 (분모, `housing_stock_v2.parquet`)

주택총조사 t년(11월 1일 기준)은 t+1년 7월 하순에 공표된다(KOSIS 최종 수정일로 확인). **t+1년 8월 행부터** 쓴다.

In [6]:
# ============================================================
# 재고: 기준월 m에 공표돼 있던 가장 최근 주택총조사
# ============================================================

census = raw["DT_1JU1501"].copy()
census = census[census["C1_NM"].isin(ALL_GU) & census["C1"].str.match(r"^11\d{3}$")]
census["value"] = pd.to_numeric(census["DT"], errors="coerce")
stock = census.pivot_table(index=["C1_NM", "PRD_DE"], columns="ITM_NM", values="value").reset_index()
stock.columns.name = None
stock = stock.rename(columns={"C1_NM": "자치구", "PRD_DE": "조사연도", "주택_계": "주택재고", "아파트": "아파트재고"})
stock["조사연도"] = stock["조사연도"].astype(int)
stock["available_from"] = pd.PeriodIndex(
    (stock["조사연도"] + 1).astype(str) + f"-{CENSUS_RELEASE_MONTH:02d}", freq="M"
)
assert stock.groupby("조사연도")["자치구"].nunique().eq(25).all()


def stock_asof(months):
    grid = pd.DataFrame([(g, m) for g in ALL_GU for m in months], columns=["자치구", "month"])
    s = stock.sort_values("available_from")
    out = []
    for gu, part in grid.groupby("자치구"):
        hist = s[s["자치구"].eq(gu)]
        idx = np.searchsorted(hist["available_from"].to_numpy(), part["month"].to_numpy(), side="right") - 1
        picked = hist.iloc[idx].reset_index(drop=True)
        out.append(pd.concat([part.reset_index(drop=True), picked[["조사연도", "주택재고", "아파트재고"]]], axis=1))
    return pd.concat(out, ignore_index=True)


stock_rm = stock_asof(MODEL_MONTHS)
display(stock_rm[stock_rm["자치구"].eq("강남구")].iloc[[0, 1, 12, 13, -1]])

,자치구,month,조사연도,주택재고,아파트재고
0,강남구,2021-07,2019,170444.0,122664.0
1,강남구,2021-08,2020,173745.0,126023.0
12,강남구,2022-07,2020,173745.0,126023.0
13,강남구,2022-08,2021,175536.0,128437.0
59,강남구,2026-06,2024,192220.0,145996.0


## 4. 누수 검증

1. **재계산 비교:** 기준월 m마다 데이터월 m-2 이후 자료와 m월 이후 공표된 주택총조사를 지우고 다시 계산한 값이 같아야 한다.
2. **검증 자체 점검:** 지연을 1개월로 줄이면 위 비교가 차이를 잡아내야 한다.

In [7]:
# ============================================================
# 1·2. 재계산 비교 + 검증 자체 점검
# ============================================================

ALL_COLS = MODEL_FEATURES + LEVEL_FEATURES


def compare(lag_used, months=MODEL_MONTHS):
    full = build_demographics(base, months, lag=lag_used).set_index(["month", "자치구"])
    bad, cols = 0, set()
    for m in months:
        cut_base = base[base.index.get_level_values("data_month") <= m - PUBLISH_LAG]
        sub = build_demographics(cut_base, pd.PeriodIndex([m]), lag=lag_used).set_index(["month", "자치구"])
        sub = sub.reindex(full.loc[[m]].index)
        diff = sub[ALL_COLS].compare(full.loc[[m], ALL_COLS])
        if not diff.empty:
            bad += 1
            cols |= set(diff.columns.get_level_values(0))
    print(f"  (lag={lag_used}) 차이 난 Feature:", sorted(cols) or "없음")
    return bad


leak = compare(PUBLISH_LAG)
print("누수 의심 기준월:", leak)
assert leak == 0

probe = MODEL_MONTHS[::6]
detected = compare(1, months=probe)
print(f"지연 1개월로 줄였을 때 차이가 난 기준월: {detected} / {len(probe)}")
assert detected > 0

# 재고: 각 기준월에 쓴 조사연도의 공표월이 기준월 이전인지
used = stock_rm.merge(stock[["자치구", "조사연도", "available_from"]], on=["자치구", "조사연도"])
assert (used["available_from"] <= used["month"]).all()

print("✅ 인구·이동·매입자·재고 Feature: 기준월에 공개되지 않은 자료 미사용, 검증이 실제 누수를 잡아냄")

  (lag=2) 차이 난 Feature: 없음
누수 의심 기준월: 0


  (lag=1) 차이 난 Feature: ['30대매입_비중_3m', '비중_30_44세', '비중_30_44세_12m변화', '비중_65세이상', '비중_65세이상_12m변화', '서울_순이동률_12m', '서울_외지인매입_비중_3m', '서울외매입_비중_3m', '세대', '세대_12m증감률', '세대원수', '세대원수_12m변화', '순이동률_12m', '순이동률_3m', '순이동률_6m', '외지인매입_비중_3m', '인구', '인구_12m증감률', '인구_3m증감률', '타구전입_비중_3m']
지연 1개월로 줄였을 때 차이가 난 기준월: 10 / 10
✅ 인구·이동·매입자·재고 Feature: 기준월에 공개되지 않은 자료 미사용, 검증이 실제 누수를 잡아냄


## 5. 단변량 점검 (Train 구간만)

V1 분류 Target(향후 3개월 구 중위 평당가 +3% 초과 상승)과의 단변량 AUC. 수준값은 참고로만 함께 본다.

In [8]:
# ============================================================
# 분류 Target (V1 정의) + 단변량 AUC
# ============================================================

from sklearn.metrics import roc_auc_score

trade = pd.read_parquet(PROCESSED_DIR / "df_fe_base.parquet", columns=["시군구명", "계약일자", "평당가"])
trade["month"] = trade["계약일자"].dt.to_period("M")
gm = (trade.groupby(["시군구명", "month"])["평당가"].median().rename("med")
      .reset_index().sort_values(["시군구명", "month"]))
g = gm.groupby("시군구명")["med"]
gm["recent3"] = g.transform(lambda x: x.rolling(3, min_periods=3).median())
future = pd.concat([g.shift(-k) for k in (1, 2, 3)], axis=1)
ok = gm["recent3"].notna() & future.notna().all(axis=1)
gm["y"] = np.where(ok, (future.median(axis=1) / gm["recent3"] - 1 > 0.03).astype(float), np.nan)

tr = demo_rm.merge(gm.rename(columns={"시군구명": "자치구"})[["자치구", "month", "y"]], on=["자치구", "month"])
tr = tr[(tr["month"] < pd.Period("2025-07", freq="M")) & tr["y"].notna()]


def auc(x, y):
    m = x.notna()
    return roc_auc_score(y[m], x[m]) if x[m].nunique() > 1 else np.nan


rows = []
for c in ALL_COLS:
    within_x = tr[c] - tr.groupby("month")[c].transform("mean")
    within_y = tr["y"] - tr.groupby("month")["y"].transform("mean")
    keep = within_y.ne(0)
    rows.append({"feature": c, "AUC": auc(tr[c], tr["y"]),
                 "월내_AUC": auc(within_x[keep], within_y[keep].gt(0).astype(int)),
                 "용도": "모델" if c in MODEL_FEATURES else "수준(제외)"})

uni = pd.DataFrame(rows)
uni["강도"] = (uni["AUC"] - 0.5).abs()
print("Train 행:", len(tr), "양성 비율:", round(tr["y"].mean(), 3))
display(uni.sort_values("강도", ascending=False).round(3))

Train 행: 1150 양성 비율: 0.357


,feature,AUC,월내_AUC,용도,강도
13,서울_순이동률_12m,0.618,0.461,모델,0.118
12,30대매입_비중_3m,0.403,0.456,모델,0.097
3,세대원수_12m변화,0.580,0.535,모델,0.080
16,세대,0.453,0.452,수준(제외),0.047
9,타구전입_비중_3m,0.536,0.536,모델,0.036
18,비중_30_44세,0.535,0.529,수준(제외),0.035
8,순이동률_12m,0.534,0.497,모델,0.034
15,인구,0.466,0.469,수준(제외),0.034
4,비중_30_44세_12m변화,0.533,0.515,모델,0.033
11,서울외매입_비중_3m,0.532,0.532,모델,0.032


## 6. 웹 표시용 최신 구별 인구 현황 (`demographics_latest_v2.parquet`)

In [9]:
# ============================================================
# 가장 최근 데이터월 기준 (모델 Feature가 아니라 단지 상세 페이지의 '동네 인구' 카드용)
# ============================================================

feat = compute_features(base).reset_index()
latest_month = feat.loc[feat["인구"].notna(), "data_month"].max()
latest = feat[feat["data_month"].eq(latest_month) & feat["지역"].ne("서울")].rename(columns={"지역": "자치구"})

# 이동·매입자는 공표가 늦어 한 달 이전 값이 최신일 수 있다
for col in ["순이동률_12m", "외지인매입_비중_3m", "30대매입_비중_3m"]:
    last = feat[feat[col].notna() & feat["지역"].ne("서울")]
    last = last[last["data_month"].eq(last["data_month"].max())].set_index("지역")
    latest[col] = latest["자치구"].map(last[col])
    latest[f"{col}_기준월"] = str(last["data_month"].max())

latest_stock = stock[stock["조사연도"].eq(stock["조사연도"].max())].set_index("자치구")
latest["아파트재고"] = latest["자치구"].map(latest_stock["아파트재고"])
latest["주택재고"] = latest["자치구"].map(latest_stock["주택재고"])
latest["재고_조사연도"] = int(stock["조사연도"].max())
latest["세대당_아파트"] = latest["아파트재고"] / latest["세대"]
latest["인구_기준월"] = str(latest_month)

latest_cols = ["자치구", "인구", "세대", "세대원수", "비중_30_44세", "비중_65세이상", "인구_12m증감률",
               "순이동률_12m", "외지인매입_비중_3m", "30대매입_비중_3m", "아파트재고", "주택재고", "세대당_아파트",
               "인구_기준월", "순이동률_12m_기준월", "외지인매입_비중_3m_기준월", "재고_조사연도"]
latest = latest[latest_cols].reset_index(drop=True)
display(latest.sort_values("순이동률_12m", ascending=False).round(3).head(10))

,자치구,인구,세대,세대원수,비중_30_44세,비중_65세이상,인구_12m증감률,순이동률_12m,외지인매입_비중_3m,30대매입_비중_3m,아파트재고,주택재고,세대당_아파트,인구_기준월,순이동률_12m_기준월,외지인매입_비중_3m_기준월,재고_조사연도
10,동대문구,353511,181376,1.949,22.930,21.281,2.594,2.519,0.704,0.468,78425.0,116994.0,0.432,2026-09,2026-08,2026-08,2025
14,서초구,420794,176375,2.386,21.770,17.949,2.016,1.187,0.562,0.233,98789.0,137444.0,0.560,2026-09,2026-08,2026-08,2025
17,송파구,648138,289024,2.243,24.490,19.202,0.373,0.297,0.593,0.318,137516.0,223437.0,0.476,2026-09,2026-08,2026-08,2025
15,성동구,274047,133689,2.050,25.147,20.090,-0.054,0.013,0.693,0.451,70657.0,93729.0,0.529,2026-09,2026-08,2026-08,2025
24,중랑구,376951,191057,1.973,22.448,24.354,-0.040,-0.020,0.769,0.282,63078.0,123361.0,0.330,2026-09,2026-08,2026-08,2025
4,관악구,477073,291409,1.637,25.890,19.857,-0.363,-0.260,0.623,0.506,59322.0,132634.0,0.204,2026-09,2026-08,2026-08,2025
1,강동구,497709,223302,2.229,23.402,20.458,-0.296,-0.312,0.599,0.358,116258.0,170045.0,0.521,2026-09,2026-08,2026-08,2025
7,금천구,222430,122345,1.818,24.407,23.259,-0.525,-0.400,0.606,0.418,33150.0,72909.0,0.271,2026-09,2026-08,2026-08,2025
13,서대문구,299254,147139,2.034,22.132,21.321,-0.612,-0.464,0.650,0.470,60366.0,107145.0,0.410,2026-09,2026-08,2026-08,2025
5,광진구,330589,172211,1.920,24.145,19.537,-0.447,-0.535,0.553,0.284,37426.0,97994.0,0.217,2026-09,2026-08,2026-08,2025


## 7. 저장

In [10]:
# ============================================================
# 저장
# - demographics_v2: 모델 Feature (구×월)
# - housing_stock_v2: 06 Merge 분모용 수준값 (구×월, 인구·세대·재고)
# ============================================================

demographics_v2 = (
    demo_rm.assign(기준월=demo_rm["month"].astype(str))
    .merge(region_month, on=["기준월", "자치구"], how="right")
    [["기준월", "자치구", "권역"] + MODEL_FEATURES]
)
assert len(demographics_v2) == 1500
assert not demographics_v2.duplicated(["기준월", "자치구"]).any()
print("모델 Feature 결측:", demographics_v2[MODEL_FEATURES].isna().sum().loc[lambda s: s > 0].to_dict() or "없음")
assert demographics_v2[MODEL_FEATURES].isna().sum().sum() == 0

levels_v2 = (
    demo_rm[["자치구", "month"] + LEVEL_FEATURES]
    .merge(stock_rm, on=["자치구", "month"])
    .assign(기준월=lambda d: d["month"].astype(str))
    .drop(columns="month")
)
assert len(levels_v2) == 1500

demographics_v2.to_parquet(PROCESSED_DIR / "demographics_v2.parquet", index=False)
levels_v2.to_parquet(PROCESSED_DIR / "housing_stock_v2.parquet", index=False)
latest.to_parquet(PROCESSED_DIR / "demographics_latest_v2.parquet", index=False)

for name in ["demographics_v2", "housing_stock_v2", "demographics_latest_v2"]:
    print(name, pd.read_parquet(PROCESSED_DIR / f"{name}.parquet").shape)

모델 Feature 결측: 없음
demographics_v2 (1500, 18)
housing_stock_v2 (1500, 10)
demographics_latest_v2 (25, 17)


## 정리

| 파일 | 단위 | 용도 |
|---|---|---|
| `demographics_v2.parquet` | 구×월 1,500행 | 31 분류, 06 Merge — 변화·이동·매입자 비중 Feature 15개 |
| `housing_stock_v2.parquet` | 구×월 1,500행 | 06 Merge 분모 (인구·세대·주택/아파트 재고, 시점 기준 적용) |
| `demographics_latest_v2.parquet` | 구 25행 | 웹 '동네 인구' 카드 |

**남은 한계**
- 한국부동산원 매입자 통계는 최초 공표값이 없어 최신 갱신값을 쓴다(비중만 사용해 영향 축소).
- 주민등록 인구는 실거주와 다를 수 있다(주소만 옮긴 경우 등).
- 주택총조사는 연 1회라 연중 재고 변화는 반영하지 못한다. 준공 공식 통계는 시도 단위까지만 있고, 원장 기반 구 단위 준공은 04에서
  공식 통계와 시점이 어긋나 제외했으므로 **구 재고는 보정하지 않고** 연 1회 값을 그대로 쓴다 (06에서 확인, 2026-10-29).